# Cross-Modal Retrieval with Heterogeneous-Graph GCNN + Supervised Hashing
**Dataset:** Flickr30k (31,783 images × 5 captions)  
**Task:** Image ⇄ Text retrieval via compact binary hash codes  
**Model:** ViT image encoder + BERT text encoder → heterogeneous graph (image / text / concept-label nodes) → GCN → tanh hashing head → sign-binarised codes  
**Evaluation:** mAP @ R, Precision@K, and CMC (Cumulative Matching Characteristics) curves for I→T and T→I

This notebook is fully self-contained: it installs dependencies, downloads Flickr30k from Hugging Face, mines concept-tag pseudo-labels from captions (Flickr30k has no native object labels), builds the heterogeneous graph, trains the model end-to-end, and renders publication-ready CMC plots.

> **Runtime:** GPU (T4 or better). `Runtime → Change runtime type → GPU`.

---

## Notebook layout
1. Environment & GPU check  
2. Imports & global config  
3. Flickr30k download + concept-label mining  
4. Pre-extract ViT image features and BERT text features  
5. Build heterogeneous graph (image — text — concept tri-partite)  
6. GCN backbone + supervised-hashing head  
7. Pairwise + quantisation + classification loss  
8. Training loop  
9. Hash-code extraction & retrieval  
10. mAP, Precision@K, CMC curves  
11. Save artifacts

## 1. Environment & GPU check

In [ ]:
# Colab runtime check
import subprocess, sys
try:
    out = subprocess.check_output(['nvidia-smi', '-L']).decode()
    print(out)
except Exception as e:
    print('No GPU detected — switch runtime to GPU. Falling back to CPU (slow).')
print('Python:', sys.version.split()[0])


In [ ]:
# Consolidate and update dependencies to resolve conflicts and ensure downloads work.
%pip install -q \
    "huggingface_hub>=0.34.0,<2.0" \
    "datasets==2.15.0" \
    "pyarrow" \
    "torch-geometric==2.5.3" \
    "transformers==4.41.2" \
    "timm==1.0.7" \
    "matplotlib==3.8.4" \
    "tqdm>=4.67.0" \
    "pandas" \
    "scikit-learn" \
    "scipy" \
    "numpy"

print('Done. NOW: Runtime → Restart session, then run from cell 1.')

## 2. Imports & global configuration

In [ ]:
import os, json, math, random, zipfile, urllib.request, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
@dataclass
class Cfg:
    # data
    data_root: str = '/content/flickr30k'
    n_train:   int = 15000      # subset size — bump up if you have time
    n_query:   int = 2000
    n_db:      int = 15000
    n_labels:  int = 68        # mined concept tags (see section 3)
    captions_per_image: int = 1

    # encoders
    vit_name:  str = 'google/vit-base-patch16-224-in21k'
    bert_name: str = 'bert-base-uncased'
    img_dim:   int = 768
    txt_dim:   int = 768

    # GCN
    gcn_hidden: int = 512
    gcn_out:    int = 256
    gcn_layers: int = 3
    dropout:    float = 0.2

    # hashing
    hash_bits:  int = 64
    alpha_pair: float = 4.0
    alpha_quant: float = 0.05
    alpha_cls:   float = 0.3

    # train
    epochs:     int = 100
    batch_size: int = 42
    lr:         float = 5e-4
    weight_decay: float = 1e-5

cfg = Cfg()
Path(cfg.data_root).mkdir(parents=True, exist_ok=True)
cfg

In [ ]:
# Wipe the partial/corrupt HF cache so the next load_dataset re-downloads cleanly
!rm -rf ~/.cache/huggingface/datasets
!rm -rf ~/.cache/huggingface/hub/datasets--nlphuji--flickr30k
!rm -rf /root/.cache/huggingface/datasets
!rm -rf /root/.cache/huggingface/hub/datasets--nlphuji--flickr30k
print('HF cache cleared.')

## 3. Download Flickr30k and mine concept-label pseudo-tags

We pull **Flickr30k** from Hugging Face (`nlphuji/flickr30k`) — 31,783 images each with 5 human-written captions. Unlike MS-COCO, Flickr30k has no per-image object categories, so we **mine 24 concept tags** by lemmatising caption text and keeping the most frequent visual nouns/verbs (`person, dog, car, water, road, ...`). An image is tagged with a concept if **any** of its 5 captions mentions a synonym of that concept. This gives a clean multi-label vector that plays the same role the 80 COCO categories played before.

In [ ]:
import os, zipfile, shutil
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

# ----------------------------------------------------------------
# Option A (preferred): Kaggle public mirror of Flickr30k
# ----------------------------------------------------------------
# This requires a free Kaggle account + API token (kaggle.json).
# 1) https://www.kaggle.com/settings/account → "Create New Token"
# 2) Upload kaggle.json to Colab via the file picker, then run:

from google.colab import files
if not Path('/root/.kaggle/kaggle.json').exists():
    print('Upload your kaggle.json (Kaggle → Settings → Create New API Token):')
    up = files.upload()                         # opens browser file picker
    Path('/root/.kaggle').mkdir(exist_ok=True)
    for fn in up:
        shutil.move(fn, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

%pip install -q kaggle

!kaggle datasets download -d hsankesara/flickr-image-dataset -p {cfg.data_root} --force

zip_path = next(Path(cfg.data_root).glob('*.zip'))
print('Downloaded:', zip_path, f'({zip_path.stat().st_size/1e6:.0f} MB)')

extract_dir = Path(cfg.data_root) / 'flickr30k_raw'
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    print('Extracted to', extract_dir)

# The Kaggle mirror layout:
#   flickr30k_images/flickr30k_images/   <- the JPGs
#   flickr30k_images/results.csv         <- captions (image_name | comment_number | comment)
img_root = next(extract_dir.rglob('flickr30k_images/*.jpg')).parent
csv_path = next(extract_dir.rglob('results.csv'))
print('Image dir :', img_root)
print('Caption csv:', csv_path)

# CSV is pipe-delimited with a stray space in the header — read robustly
df = pd.read_csv(csv_path, sep='|', engine='python',
                 encoding='utf-8', encoding_errors='replace')
df.columns = [c.strip() for c in df.columns]
print('Caption rows:', len(df), '| columns:', df.columns.tolist())
df.head(3)

In [ ]:
# Group the 5 captions per image, then sample the slice we need
df['comment'] = df['comment'].astype(str).str.strip()
grouped = (df.groupby('image_name')['comment']
             .apply(lambda s: s.tolist()[:5])
             .reset_index())
print('Unique images:', len(grouped))

n_need = cfg.n_train + cfg.n_query
sub = grouped.sample(frac=1, random_state=SEED).reset_index(drop=True).head(n_need)

raw_records = []
for k, row in tqdm(sub.iterrows(), total=len(sub), desc='Building records'):
    fname = img_root / row['image_name']
    if not fname.exists():
        continue
    caps = [c for c in row['comment'] if c and c != 'nan']
    if not caps:
        continue
    raw_records.append({
        'image_id': int(Path(row['image_name']).stem.lstrip('0') or '0'),
        'file':     str(fname),
        'all_caps': caps[:5],
        'caption':  caps[0],
    })

print('Built', len(raw_records), 'records')
print('Example:', {k: v if k != 'all_caps' else v[:2]
                   for k, v in raw_records[0].items()})


import re
from collections import Counter

# --- Concept vocabulary (24 visual concepts grouped with synonyms) -----
# CONCEPT_VOCAB = {
#     'person'   : ['person','people','man','woman','boy','girl','child','kid','guy','lady'],
#     'group'    : ['group','crowd','team','family','couple'],
#     'dog'      : ['dog','puppy'],
#     'cat'      : ['cat','kitten'],
#     'horse'    : ['horse','pony'],
#     'bird'     : ['bird','seagull','duck'],
#     'animal'   : ['animal','cow','sheep','goat','elephant','bear','deer'],
#     'vehicle'  : ['car','truck','bus','motorcycle','bike','bicycle','vehicle','scooter'],
#     'boat'     : ['boat','ship','kayak','canoe'],
#     'building' : ['building','house','tower','church','school','store','shop'],
#     'street'   : ['street','road','sidewalk','highway','path','trail'],
#     'water'    : ['water','river','lake','ocean','sea','pool','beach','wave'],
#     'snow'     : ['snow','ice','snowy','snowboard','ski','skiing'],
#     'grass'    : ['grass','field','lawn','meadow','park'],
#     'mountain' : ['mountain','hill','cliff','rock','rocks'],
#     'sky'      : ['sky','cloud','clouds','sunset','sunrise'],
#     'food'     : ['food','meal','pizza','sandwich','cake','fruit','vegetable'],
#     'sport'    : ['sport','game','ball','football','basketball','soccer','tennis','baseball','skateboard','surf','surfing'],
#     'music'    : ['music','guitar','drum','piano','singer','band','microphone'],
#     'clothing' : ['shirt','dress','jacket','hat','helmet','uniform','suit'],
#     'indoor'   : ['indoor','room','kitchen','bedroom','office','restaurant','bar'],
#     'tree'     : ['tree','forest','woods','jungle'],
#     'work'     : ['working','worker','construction','tool','painting','digging'],
#     'sit'      : ['sitting','seated','sat','standing','walking','running','jumping'],
# }
# CONCEPTS = list(CONCEPT_VOCAB.keys())
# assert len(CONCEPTS) == cfg.n_labels, f'cfg.n_labels must be {len(CONCEPTS)}'
# concept2idx = {c: i for i, c in enumerate(CONCEPTS)}
# idx_to_cat  = {i: c for c, i in concept2idx.items()}     # for the demo cells

# # Build a fast lookup: word -> concept index
# word2concept = {}
# for concept, words in CONCEPT_VOCAB.items():
#     for w in words:
#         word2concept[w.lower()] = concept2idx[concept]
# ============================================================
# Expanded concept vocabulary for Flickr30k (48 concepts)
# Curated from analysis of caption frequency in Flickr30k.
# Each concept groups morphological/synonym variants so the
# mining stays robust without needing an NLP lemmatiser.
# ============================================================
CONCEPT_VOCAB = {
    # ---- People & roles ----
    'man'        : ['man','men','guy','gentleman','male','boyfriend','husband','father','dad','son','brother'],
    'woman'      : ['woman','women','lady','female','girlfriend','wife','mother','mom','daughter','sister'],
    'child'      : ['child','children','kid','kids','baby','toddler','infant','youngster'],
    'boy'        : ['boy','boys','teenager','teen'],
    'girl'       : ['girl','girls'],
    'group'      : ['group','crowd','team','family','couple','pair','audience','people'],
    'worker'     : ['worker','workers','construction','builder','farmer','fisherman','mechanic','chef','cook','waiter','soldier','officer','police','policeman'],
    'performer'  : ['performer','dancer','singer','musician','actor','model','speaker','presenter'],
    'athlete'    : ['athlete','player','runner','swimmer','climber','skier','snowboarder','surfer','skater','cyclist','biker'],

    # ---- Animals ----
    'dog'        : ['dog','dogs','puppy','puppies'],
    'cat'        : ['cat','cats','kitten','kittens'],
    'horse'      : ['horse','horses','pony','ponies'],
    'bird'       : ['bird','birds','seagull','duck','ducks','goose','chicken','rooster','parrot','eagle','owl'],
    'farm_animal': ['cow','cows','sheep','goat','goats','pig','pigs','bull'],
    'wild_animal': ['elephant','bear','deer','lion','tiger','monkey','zebra','giraffe','wolf','fox','squirrel','rabbit'],
    'fish'       : ['fish','shark','dolphin','whale'],

    # ---- Vehicles & transport ----
    'car'        : ['car','cars','automobile','sedan','taxi','cab'],
    'truck'      : ['truck','trucks','van','pickup','lorry'],
    'bus'        : ['bus','buses','tram','trolley'],
    'motorcycle' : ['motorcycle','motorbike','scooter','moped'],
    'bicycle'    : ['bicycle','bicycles','bike','bikes','cyclist','cycling'],
    'boat'       : ['boat','boats','ship','ships','yacht','kayak','canoe','raft','sailboat'],
    'aircraft'   : ['plane','airplane','aircraft','jet','helicopter','glider'],
    'train'      : ['train','trains','subway','metro','rail','railway'],

    # ---- Built environment ----
    'building'   : ['building','buildings','house','houses','tower','skyscraper','apartment'],
    'religious'  : ['church','cathedral','temple','mosque','synagogue'],
    'commercial' : ['store','shop','restaurant','cafe','bar','market','mall','stall'],
    'street'     : ['street','streets','road','roads','sidewalk','pavement','highway','alley','intersection','crosswalk'],
    'path'       : ['path','trail','walkway','bridge','tunnel','staircase','stairs','steps'],
    'indoor'     : ['indoor','indoors','room','kitchen','bedroom','bathroom','office','hallway','hall','lobby','classroom','gym','studio'],

    # ---- Nature & environment ----
    'water'      : ['water','river','lake','pond','ocean','sea','bay','harbor','pool','beach','shore','wave','waves','stream','creek','waterfall','fountain'],
    'snow'       : ['snow','snowy','snowing','ice','icy','frozen','glacier'],
    'grass'      : ['grass','grassy','field','fields','lawn','meadow','park','pasture'],
    'mountain'   : ['mountain','mountains','hill','hills','cliff','cliffs','rock','rocks','rocky','canyon','valley'],
    'sky'        : ['sky','cloud','clouds','cloudy','sunset','sunrise','dusk','dawn','rainbow','sunny'],
    'tree'       : ['tree','trees','forest','woods','jungle','bush','bushes','hedge'],
    'flower'     : ['flower','flowers','garden','blossom','rose','tulip'],
    'desert'     : ['desert','sand','sandy','dune','dunes'],
    'rain'       : ['rain','raining','rainy','wet','puddle','storm','stormy'],
    'fire'       : ['fire','flame','flames','smoke','bonfire','campfire'],

    # ---- Activities & sports ----
    'ball_sport' : ['football','soccer','basketball','baseball','volleyball','tennis','golf','cricket','rugby','ball','goal','net'],
    'water_sport': ['surf','surfing','surfer','swim','swimming','diving','kayaking','rowing','sailing','rafting'],
    'snow_sport' : ['ski','skiing','snowboard','snowboarding','sled','sledding','skating'],
    'wheel_sport': ['skate','skateboard','skateboarding','rollerblade','bmx'],
    'climb'      : ['climb','climbing','climber','hiking','hiker','rappelling'],
    'dance'      : ['dance','dancing','ballet','choreography'],
    'music_act'  : ['music','musician','singing','concert','band','orchestra','guitar','drum','drums','piano','violin','microphone','stage'],
    'art'        : ['paint','painting','painter','draw','drawing','sculpture','mural','graffiti'],
    'work_act'   : ['working','digging','building','repairing','cleaning','cooking','farming','fishing'],
    'play'       : ['playing','game','toy','toys','playground','swing','slide'],

    # ---- Posture & motion (verbs that survive without a lemmatiser) ----
    'sit'        : ['sit','sitting','seated','sat','perched'],
    'stand'      : ['stand','standing','stood'],
    'walk'       : ['walk','walking','walked','strolling','stroll'],
    'run'        : ['run','running','sprinting','jogging','jogger'],
    'jump'       : ['jump','jumping','leaping','leap','hopping'],
    'lie'        : ['lying','lying down','laying','reclining','sleeping','asleep'],
    'hold'       : ['holding','carrying','clutching','grasping','gripping'],
    'look'       : ['looking','watching','staring','gazing','observing'],

    # ---- Objects (frequent in Flickr captions) ----
    'food'       : ['food','meal','eating','pizza','sandwich','burger','cake','bread','fruit','vegetable','salad','soup','pasta','rice'],
    'drink'      : ['drink','drinking','beer','wine','coffee','tea','juice','water bottle','bottle','cup','mug','glass'],
    'clothing'   : ['shirt','tshirt','dress','jacket','coat','hat','cap','helmet','uniform','suit','tie','scarf','glove','gloves','boot','boots','shoe','shoes','sneakers','jeans','pants','shorts','skirt','sweater'],
    'accessory'  : ['bag','backpack','purse','handbag','umbrella','glasses','sunglasses','watch','jewelry'],
    'tool'       : ['tool','tools','hammer','knife','axe','rope','ladder','shovel','rake','wheelbarrow','machine'],
    'weapon'     : ['gun','rifle','pistol','sword','bow','arrow'],
    'electronic' : ['phone','cellphone','smartphone','laptop','computer','camera','television','tv','tablet','headphones'],
    'book'       : ['book','books','newspaper','magazine','sign','poster','banner'],

    # ---- Settings / scene context ----
    'urban'      : ['city','urban','downtown','plaza','square','rooftop'],
    'rural'      : ['rural','countryside','farm','barn','village','ranch'],
    'event'      : ['parade','festival','wedding','party','ceremony','protest','rally','march','marathon','race','competition'],

    # ---- Visual modifiers (high-signal in captions) ----
    'night'      : ['night','nighttime','evening','dark','lantern','candle','firework','fireworks'],
    'colorful'   : ['colorful','colourful','bright','neon','rainbow-colored','painted'],
}

CONCEPTS    = list(CONCEPT_VOCAB.keys())
concept2idx = {c: i for i, c in enumerate(CONCEPTS)}
idx_to_cat  = {i: c for c, i in concept2idx.items()}

# Build fast word→concept lookup with simple plural handling
word2concept = {}
for concept, words in CONCEPT_VOCAB.items():
    for w in words:
        wl = w.lower()
        word2concept[wl] = concept2idx[concept]
        # Auto-add plural form unless it's already a multi-word phrase
        if ' ' not in wl and not wl.endswith('s'):
            word2concept[wl + 's'] = concept2idx[concept]

# Update cfg.n_labels to match — IMPORTANT: re-run the model cell after this
cfg.n_labels = len(CONCEPTS) # ADDED THIS LINE
print(f'Vocabulary size: {len(CONCEPTS)} concepts')
print(f'Lookup entries : {len(word2concept)} (incl. plurals)')
print(f'\n>>> Set cfg.n_labels = {len(CONCEPTS)} and re-run the model cell.')
WORD_RE = re.compile(r"[A-Za-z']+")

def mine_labels(captions):
    """Return a multi-hot label vector by scanning all captions for concept words."""
    label = np.zeros(cfg.n_labels, dtype=np.float32)
    for cap in captions:
        for w in WORD_RE.findall(cap.lower()):
            # crude lemmatisation — strip plural 's'
            for variant in (w, w.rstrip('s')):
                if variant in word2concept:
                    label[word2concept[variant]] = 1.0
                    break
    return label

# Apply to all materialised records
records = []
for r in raw_records:
    label = mine_labels(r['all_caps'])
    if label.sum() == 0:
        continue                # drop captions with no recognised concepts (rare)
    records.append({**r, 'label': label})

print(f'# records after concept mining: {len(records)} / {len(raw_records)}')

# Show concept frequency so you can verify the mining is sensible
freq = Counter()
for r in records:
    for i in np.where(r['label'] > 0)[0]:
        freq[idx_to_cat[i]] += 1
print('\nConcept frequencies:')
for c, n in freq.most_common():
    print(f'  {c:<10} {n:>4}  ({n/len(records)*100:>4.1f}%)')

print('\nExample record:', {
    'file':    Path(records[0]['file']).name,
    'caption': records[0]['caption'],
    'concepts':[idx_to_cat[i] for i in np.where(records[0]['label'] > 0)[0]],
})


In [ ]:
assert len(records) > 14000, f'Expected ~15000+ records, got {len(records)}'

In [ ]:
print('cfg.n_train + cfg.n_query =', cfg.n_train + cfg.n_query)
print('len(raw_records)          =', len(raw_records))
print('len(records)              =', len(records))
print('Number of unique images on disk:',
      len(list((Path(cfg.data_root)/'flickr30k_raw'/'flickr30k_images'/
                'flickr30k_images').glob('*.jpg'))))

Let's inspect the structure of `records` list.

## 4. Pre-extract ViT image features and BERT text features

Encoding 5 k images with ViT and 5 k captions with BERT once and caching to tensors keeps the rest of training fast and lets the GCN focus purely on cross-modal alignment. We use the `[CLS]` token of each encoder (768-d).

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
from PIL import Image

@torch.no_grad()
def extract_image_feats(records, batch=32):
    proc  = AutoImageProcessor.from_pretrained(cfg.vit_name)
    model = AutoModel.from_pretrained(cfg.vit_name).to(DEVICE).eval()
    feats = torch.empty(len(records), cfg.img_dim)
    for i in tqdm(range(0, len(records), batch), desc='ViT'):
        chunk = records[i:i+batch]
        imgs  = [Image.open(r['file']).convert('RGB') for r in chunk]
        px    = proc(images=imgs, return_tensors='pt')['pixel_values'].to(DEVICE)
        out   = model(pixel_values=px).last_hidden_state[:, 0]  # CLS
        feats[i:i+len(chunk)] = out.cpu()
    del model; torch.cuda.empty_cache()
    return feats

@torch.no_grad()
def extract_text_feats(records, batch=64):
    tok   = AutoTokenizer.from_pretrained(cfg.bert_name)
    model = AutoModel.from_pretrained(cfg.bert_name).to(DEVICE).eval()
    feats = torch.empty(len(records), cfg.txt_dim)
    for i in tqdm(range(0, len(records), batch), desc='BERT'):
        chunk = records[i:i+batch]
        enc = tok([r['caption'] for r in chunk],
                  padding=True, truncation=True, max_length=64,
                  return_tensors='pt').to(DEVICE)
        out = model(**enc).last_hidden_state[:, 0]              # CLS
        feats[i:i+len(chunk)] = out.cpu()
    del model; torch.cuda.empty_cache()
    return feats

cache = Path(cfg.data_root)/'feat_cache.pt'
# Delete cache if n_labels has changed to force regeneration with correct label dimensions
if cache.exists():
    cached_blob = torch.load(cache, map_location='cpu')
    if 'lbl' in cached_blob and cached_blob['lbl'].shape[1] != cfg.n_labels:
        print(f"Deleting old cache: cfg.n_labels changed from {cached_blob['lbl'].shape[1]} to {cfg.n_labels}")
        cache.unlink() # Delete the cache file

if cache.exists():
    blob = torch.load(cache, map_location='cpu')
    img_feats, txt_feats, labels = blob['img'], blob['txt'], blob['lbl']
    print('Loaded cached features.')
else:
    # Add a check to ensure records is not empty
    if not records:
        raise ValueError("The 'records' list is empty. Please ensure the previous cell (concept mining) ran successfully and populated 'records' before attempting to extract features.")
    img_feats = extract_image_feats(records)
    txt_feats = extract_text_feats(records)
    labels    = torch.tensor(np.stack([r['label'] for r in records]))
    torch.save({'img':img_feats,'txt':txt_feats,'lbl':labels}, cache)

print('img_feats', img_feats.shape, 'txt_feats', txt_feats.shape, 'labels', labels.shape)


In [ ]:
# Train / query split (query is held out from training graph)
N = img_feats.size(0)
perm = torch.randperm(N, generator=torch.Generator().manual_seed(SEED))
q_idx = perm[:cfg.n_query]
t_idx = perm[cfg.n_query:cfg.n_query + cfg.n_train]

train_img, train_txt, train_lbl = img_feats[t_idx], txt_feats[t_idx], labels[t_idx]
query_img, query_txt, query_lbl = img_feats[q_idx], txt_feats[q_idx], labels[q_idx]

# Database == training pool (standard cross-modal-hashing protocol)
db_img, db_txt, db_lbl = train_img, train_txt, train_lbl

print('Train:', train_img.shape[0], '| Query:', query_img.shape[0],
      '| DB:', db_img.shape[0])


## 5. Build a heterogeneous graph

Three node types:
- **image** nodes (one per training image, ViT feature)
- **text** nodes (one per training caption, BERT feature)
- **label** nodes (80 object categories, learnable embedding)

Edge types:
- `image ↔ text` — paired (positive) cross-modal edges  
- `image ↔ label`, `text ↔ label` — multi-label assignment edges  
- `image ↔ image`, `text ↔ text` — kNN intra-modal edges (cosine, k=10)

We materialise it as a `torch_geometric.data.HeteroData` object so PyG's `HeteroConv` handles message passing per relation.

In [ ]:
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GCNConv, SAGEConv
from sklearn.metrics.pairwise import cosine_similarity

def knn_edges(x: torch.Tensor, k: int = 10) -> torch.Tensor:
    '''Computes k-NN edges for a given feature matrix x.'''
    # Move the tensor to CPU before converting to numpy
    sim = cosine_similarity(x.cpu().numpy())
    np.fill_diagonal(sim, -1)
    nn_idx = np.argpartition(-sim, kth=k, axis=1)[:, :k]
    src = np.repeat(np.arange(x.size(0)), k)
    dst = nn_idx.reshape(-1)
    e = np.stack([src, dst]); e = np.concatenate([e, e[::-1]], axis=1)  # undirected
    return torch.from_numpy(e).long()

def build_hetero(img_x, txt_x, lbl_y, n_labels=cfg.n_labels, k=10):
    g = HeteroData()
    g['image'].x = img_x
    g['text' ].x = txt_x
    g['label'].x = torch.randn(n_labels, img_x.size(1)) * 0.01  # learnable init

    n = img_x.size(0)

    # paired cross-modal edges
    pair = torch.arange(n)
    g['image','pair','text'].edge_index = torch.stack([pair, pair])
    g['text','pair','image'].edge_index = torch.stack([pair, pair])

    # label-assignment edges
    img_e = lbl_y.nonzero(as_tuple=False).t()                       # [2, E]
    g['image','has','label'].edge_index = img_e
    g['label','of'  ,'image'].edge_index = img_e.flip(0)
    g['text' ,'has','label'].edge_index = img_e                     # caption shares labels
    g['label','of'  ,'text' ].edge_index = img_e.flip(0)

    # kNN intra-modal edges
    g['image','knn','image'].edge_index = knn_edges(g['image'].x, k)
    g['text' ,'knn','text' ].edge_index = knn_edges(g['text'].x, k)
    return g

graph = build_hetero(train_img, train_txt, train_lbl).to(DEVICE)
print(graph)

## 6. GCN backbone + supervised hashing head

In [ ]:
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.nn import HeteroConv, SAGEConv, GCNConv

class HeteroGCN(nn.Module):
    def __init__(self, in_dim, hidden, out, layers=2, dropout=0.2):
        super().__init__()
        self.proj = nn.ModuleDict({
            nt: nn.Sequential(nn.Linear(in_dim, hidden), nn.LayerNorm(hidden))
            for nt in ['image', 'text', 'label']
        })
        self.norms = nn.ModuleList()
        self.convs = nn.ModuleList()
        for li in range(layers):
            d_in  = hidden
            d_out = out if li == layers - 1 else hidden
            self.convs.append(HeteroConv({
                ('image','pair','text' ): SAGEConv(d_in, d_out),
                ('text' ,'pair','image'): SAGEConv(d_in, d_out),
                ('image','has' ,'label'): SAGEConv(d_in, d_out),
                ('label','of'  ,'image'): SAGEConv(d_in, d_out),
                ('text' ,'has' ,'label'): SAGEConv(d_in, d_out),
                ('label','of'  ,'text' ): SAGEConv(d_in, d_out),
                ('image','knn' ,'image'): SAGEConv(d_in, d_out),
                ('text' ,'knn' ,'text' ): SAGEConv(d_in, d_out),
            }, aggr='mean'))
            self.norms.append(nn.ModuleDict({
                nt: nn.LayerNorm(d_out) for nt in ['image','text','label']
            }))
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        x = {k: self.proj[k](v) for k, v in x_dict.items()}
        for conv, norm in zip(self.convs, self.norms):
            x_new = conv(x, edge_index_dict)
            x_new = {k: F.dropout(F.relu(norm[k](v)), p=self.dropout,
                                  training=self.training)
                     for k, v in x_new.items()}
            # residual where shapes match
            x = {k: x_new[k] + x[k] if x[k].shape == x_new[k].shape else x_new[k]
                 for k in x_new}
        return x


class CMHashNet(nn.Module):
    '''Heterogeneous-graph encoder + per-modality hashing head + classifier.'''
    def __init__(self, cfg):
        super().__init__()
        self.gcn = HeteroGCN(cfg.img_dim, cfg.gcn_hidden, cfg.gcn_out,
                             cfg.gcn_layers, cfg.dropout)
        # Hashing head: BN keeps pre-tanh activations in a healthy range,
        # which is the single most important change to prevent collapse.
        self.hash_img = nn.Sequential(
            nn.Linear(cfg.gcn_out, cfg.hash_bits),
            nn.BatchNorm1d(cfg.hash_bits),
        )
        self.hash_txt = nn.Sequential(
            nn.Linear(cfg.gcn_out, cfg.hash_bits),
            nn.BatchNorm1d(cfg.hash_bits),
        )
        self.cls = nn.Linear(cfg.hash_bits, cfg.n_labels)

    def forward(self, x_dict, edge_index_dict):
        h = self.gcn(x_dict, edge_index_dict)
        # Scaled tanh — multiply by a temperature < 1 so we live in the
        # responsive part of the curve, not the saturated tails.
        z_img = torch.tanh(self.hash_img(h['image']) * 0.5)
        z_txt = torch.tanh(self.hash_txt(h['text' ]) * 0.5)
        return z_img, z_txt, self.cls(z_img), self.cls(z_txt)

model = CMHashNet(cfg).to(DEVICE)
print(f'# parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')

## 7. Loss functions

**Pairwise similarity loss** — the binary "should-match" target $S_{ij} \in \{0,1\}$ is built from label co-occurrence (≥1 shared category ⇒ similar). The continuous code inner-product is pulled toward $b \cdot S$:

$$\mathcal{L}_{\text{pair}} = \frac{1}{N^2}\sum_{i,j}\Bigl(\frac{1}{b}\,z_i^{\top} z_j - (2 S_{ij} - 1)\Bigr)^2$$

**Quantisation loss** — pushes tanh outputs toward $\{-1,+1\}$:

$$\mathcal{L}_{\text{quant}} = \|z - \operatorname{sign}(z)\|_2^2$$

**Multi-label classification** — keeps codes label-discriminative:

$$\mathcal{L}_{\text{cls}} = \text{BCE}(\sigma(W z), y)$$

Total: $\mathcal{L} = \alpha_p \mathcal{L}_{\text{pair}} + \alpha_q \mathcal{L}_{\text{quant}} + \alpha_c \mathcal{L}_{\text{cls}}$.

In [ ]:
def info_nce(z_a, z_b, temp=0.1):
    """Symmetric InfoNCE on L2-normalised codes. Paired samples on the diagonal."""
    a = F.normalize(z_a, dim=1)
    b = F.normalize(z_b, dim=1)
    logits = (a @ b.t()) / temp                          # [N, N]
    labels = torch.arange(z_a.size(0), device=z_a.device)
    return 0.5 * (F.cross_entropy(logits, labels) +
                  F.cross_entropy(logits.t(), labels))

def supervised_contrastive(z, y, temp=0.1):
    """Pull together samples sharing ≥1 label, push apart the rest."""
    z = F.normalize(z, dim=1)
    sim = (z @ z.t()) / temp
    sim = sim - sim.amax(dim=1, keepdim=True).detach()    # numerical stability
    pos = (y @ y.t() > 0).float()
    pos.fill_diagonal_(0)
    exp = sim.exp()
    exp.fill_diagonal_(0)
    log_prob = sim - (exp.sum(1, keepdim=True) + 1e-8).log()
    pos_per = pos.sum(1).clamp(min=1)
    return -((pos * log_prob).sum(1) / pos_per).mean()

def quant_loss(z):
    return ((z.abs() - 1) ** 2).mean()

def balance_loss(z):
    return z.mean(0).pow(2).mean()

def decorrelation_loss(z):
    """Cross-correlation penalty (Barlow-Twins style) — sums over bit-pairs
    instead of averaging, so the gradient actually has bite."""
    zn = (z - z.mean(0)) / (z.std(0) + 1e-5)
    C  = (zn.t() @ zn) / z.size(0)                        # [bits, bits]
    on  = ((torch.diag(C) - 1) ** 2).sum()                # diag → 1
    off = (C - torch.diag(torch.diag(C))).pow(2).sum()    # off-diag → 0
    return on + 5e-3 * off                                # off-diag weight inside

def total_loss(z_i, z_t, lo_i, lo_t, y, cfg):
    # 1. Cross-modal contrastive: paired (image, text) must be closest
    L_nce = info_nce(z_i, z_t, temp=0.1)
    # 2. Supervised contrastive within each modality (uses label info)
    L_sup = supervised_contrastive(z_i, y) + supervised_contrastive(z_t, y)
    # 3. Hashing-specific regularisers
    L_q = quant_loss(z_i) + quant_loss(z_t)
    L_b = balance_loss(z_i) + balance_loss(z_t)
    L_d = decorrelation_loss(z_i) + decorrelation_loss(z_t)
    # 4. Multi-label classifier (auxiliary, kept light)
    L_c = (F.binary_cross_entropy_with_logits(lo_i, y) +
           F.binary_cross_entropy_with_logits(lo_t, y))
    L = (4.0 * L_nce +
         1.0 * L_sup +
         cfg.alpha_quant * L_q +
         1.0 * L_b +
         1.0 * L_d +
         cfg.alpha_cls * L_c)
    return L, {'nce': L_nce.item(), 'sup': L_sup.item(),
               'quant': L_q.item(), 'bal': L_b.item(),
               'dec': L_d.item(),   'cls': L_c.item()}

## 8. Training loop (full-batch graph)

In [ ]:
def supervised_contrastive(z, y, temp=0.1):
    """Pull together samples sharing ≥1 label, push apart the rest."""
    z = F.normalize(z, dim=1)
    sim = (z @ z.t()) / temp
    sim = sim - sim.amax(dim=1, keepdim=True).detach()    # numerical stability

    pos = (y @ y.t() > 0).float()
    # Create a new tensor for `pos` to avoid in-place modification
    pos_new = pos.clone()
    pos_new.fill_diagonal_(0)

    # Create a new tensor for `exp` to avoid in-place modification
    exp_original = sim.exp()
    exp_new = exp_original.clone()
    exp_new.fill_diagonal_(0)

    log_prob = sim - (exp_new.sum(1, keepdim=True) + 1e-8).log()
    pos_per = pos_new.sum(1).clamp(min=1)
    return -((pos_new * log_prob).sum(1) / pos_per).mean()

optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                          weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg.epochs)

train_lbl_d = train_lbl.to(DEVICE)
history = {'loss':[], 'nce':[], 'sup':[], 'quant':[],
           'bal':[], 'dec':[], 'cls':[]}

for ep in range(1, cfg.epochs + 1):
    model.train()
    z_i, z_t, lo_i, lo_t = model(graph.x_dict, graph.edge_index_dict)

    warm = min(1.0, ep / (0.3 * cfg.epochs))
    cfg_dyn = type(cfg)(**{**cfg.__dict__, 'alpha_quant': cfg.alpha_quant * warm})

    loss, parts = total_loss(z_i, z_t, lo_i, lo_t, train_lbl_d, cfg_dyn)

    optim.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optim.step(); sched.step()

    history['loss'].append(loss.item())
    for k, v in parts.items(): history[k].append(v)

    if ep % 5 == 0 or ep == 1 or ep == cfg.epochs:
        with torch.no_grad():
            sb = z_i.sign().float()
            bal = sb.mean(0).abs().mean().item()
            uniq = len(torch.unique(sb, dim=0))
        print(f'ep {ep:03d} | L {loss.item():.2f} | nce {parts["nce"]:.2f} | '
              f'sup {parts["sup"]:.2f} | dec {parts["dec"]:.2f} | '
              f'bal {bal:.3f} | unique-codes {uniq}/{z_i.size(0)}')

In [ ]:
# Loss curves
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(history['loss']); ax[0].set_title('Total loss'); ax[0].set_xlabel('epoch')
ax[1].plot(history['nce'],  label='pair')
ax[1].plot(history['quant'], label='quant')
ax[1].plot(history['cls'],   label='cls')
ax[1].set_title('Loss components'); ax[1].legend(); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig('loss_curves.png', dpi=140); plt.show()

## 9. Hash-code extraction for query and database

For the query set we don't include the points in the GCN message passing — we encode them via an *inductive* path: project raw ViT/BERT features through the trained `proj` + skip the graph convs (using a small fallback MLP path is common, but to keep training-eval consistent we re-run the GCN with the query nodes inserted and *no* label edges, then read out their codes).

In [ ]:
@torch.no_grad()
def encode_codes_continuous(model, base_graph, q_img, q_txt, lbl_for_db):
    """Same as encode_codes but ALSO returns continuous tanh outputs for re-ranking."""
    model.eval()
    n_db = base_graph['image'].x.size(0)

    g = HeteroData()
    g['image'].x = torch.cat([base_graph['image'].x.cpu(), q_img], 0)
    g['text' ].x = torch.cat([base_graph['text' ].x.cpu(), q_txt], 0)
    g['label'].x = base_graph['label'].x.cpu()

    pair_db = torch.arange(n_db)
    g['image','pair','text'].edge_index = torch.stack([pair_db, pair_db])
    g['text','pair','image'].edge_index = torch.stack([pair_db, pair_db])

    img_e = lbl_for_db.nonzero(as_tuple=False).t()
    g['image','has','label'].edge_index = img_e
    g['label','of'  ,'image'].edge_index = img_e.flip(0)
    g['text' ,'has','label'].edge_index = img_e
    g['label','of'  ,'text' ].edge_index = img_e.flip(0)
    g['image','knn','image'].edge_index = knn_edges(g['image'].x, k=10)
    g['text' ,'knn','text' ].edge_index = knn_edges(g['text' ].x, k=10)

    g = g.to(DEVICE)
    z_i_cont, z_t_cont, _, _ = model(g.x_dict, g.edge_index_dict)
    z_i_bin, z_t_bin = z_i_cont.sign(), z_t_cont.sign()
    return (z_i_bin[:n_db],  z_t_bin[:n_db],
            z_i_bin[n_db:],  z_t_bin[n_db:],
            z_i_cont[:n_db], z_t_cont[:n_db],
            z_i_cont[n_db:], z_t_cont[n_db:])

(db_i, db_t, q_i, q_t,
 db_ic, db_tc, q_ic, q_tc) = encode_codes_continuous(
    model, graph, query_img, query_txt, train_lbl)

print('Binary codes:', db_i.shape, '| Continuous codes:', db_ic.shape)

In [ ]:
def retrieve_rerank(q_bin, q_cont, db_bin, db_cont, top_k=10,
                    coarse_pool=200, alpha=0.7):
    """Two-stage retrieval: Hamming top-N, then re-rank by cosine on continuous codes.
    alpha mixes the two scores; higher = trust Hamming more."""
    # Stage 1 — coarse Hamming shortlist
    h_dist  = hamming(q_bin, db_bin).squeeze(0)            # [N]
    pool    = h_dist.argsort()[:coarse_pool]               # top-N candidates

    # Stage 2 — fine cosine re-rank on continuous codes
    q_c     = F.normalize(q_cont, dim=1)
    d_c     = F.normalize(db_cont[pool], dim=1)
    cos     = (q_c @ d_c.t()).squeeze(0)                   # [coarse_pool], in [-1,1]

    # Combine: lower h_dist is better, higher cos is better
    h_norm  = h_dist[pool].float() / q_bin.size(1)         # → [0,1]
    score   = alpha * h_norm - (1 - alpha) * cos           # lower is better
    final   = pool[score.argsort()[:top_k]]
    return final, h_dist[final], cos[score.argsort()[:top_k]]

## 10. Retrieval evaluation: mAP, Precision@K, CMC

We compute Hamming distance between every query code and every database code, sort, and use **multi-label ground truth** (a retrieved item is "relevant" if it shares ≥1 category with the query).

- **mAP@R** — mean Average Precision over the top-R results  
- **P@K** — precision among the top-K returned items  
- **CMC@k** — fraction of queries whose top-k ranks contain *at least one* relevant item (the standard re-identification / retrieval matching curve)

In [ ]:
def hamming(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    '''a:[Q,b], b:[N,b] in {-1,+1}; returns [Q,N] integer Hamming distance.'''
    return (a.size(1) - a @ b.t()) / 2

def relevant_mask(q_lbl: torch.Tensor, db_lbl: torch.Tensor) -> torch.Tensor:
    return (q_lbl @ db_lbl.t() > 0).float()

def map_at_r(rel_sorted: torch.Tensor, R: int) -> float:
    rel = rel_sorted[:, :R]
    cum = rel.cumsum(1)
    prec = cum / torch.arange(1, R+1, device=rel.device).float()
    denom = rel.sum(1).clamp(min=1)
    return ((prec * rel).sum(1) / denom).mean().item()

def precision_at_k(rel_sorted: torch.Tensor, K: int) -> float:
    return rel_sorted[:, :K].mean(1).mean().item()

def cmc_curve(rel_sorted: torch.Tensor, max_k: int) -> np.ndarray:
    '''CMC@k = P(at least one relevant in top-k).'''
    cum_hit = (rel_sorted[:, :max_k].cumsum(1) > 0).float()
    return cum_hit.mean(0).cpu().numpy()

def eval_pair(q_codes, db_codes, q_lbl, db_lbl, R=1000, K=100, max_k=200):
    dist = hamming(q_codes, db_codes)
    order = dist.argsort(dim=1)
    rel = relevant_mask(q_lbl.to(DEVICE), db_lbl.to(DEVICE))
    rel_sorted = torch.gather(rel, 1, order)
    return {
        'mAP@R': map_at_r(rel_sorted, min(R, db_codes.size(0))),
        f'P@{K}': precision_at_k(rel_sorted, K),
        'cmc':    cmc_curve(rel_sorted, max_k),
    }

q_lbl_d  = query_lbl.to(DEVICE)
db_lbl_d = train_lbl.to(DEVICE)

print('Evaluating Image → Text ...')
i2t = eval_pair(q_i, db_t, q_lbl_d, db_lbl_d)
print('Evaluating Text → Image ...')
t2i = eval_pair(q_t, db_i, q_lbl_d, db_lbl_d)

print(f"I→T  mAP@1000 = {i2t['mAP@R']:.4f}   P@100 = {i2t['P@100']:.4f}")
print(f"T→I  mAP@1000 = {t2i['mAP@R']:.4f}   P@100 = {t2i['P@100']:.4f}")


In [ ]:
# CMC curves
fig, ax = plt.subplots(figsize=(7,5))
k = np.arange(1, len(i2t['cmc'])+1)
ax.plot(k, i2t['cmc'], label=f"I→T  (mAP={i2t['mAP@R']:.3f})", lw=2)
ax.plot(k, t2i['cmc'], label=f"T→I  (mAP={t2i['mAP@R']:.3f})", lw=2)
ax.set_xlabel('Rank k'); ax.set_ylabel('CMC@k = P(hit in top-k)')
ax.set_title(f'Proposed Method CMC — {cfg.hash_bits}-bit codes')
ax.set_xscale('log'); ax.grid(True, which='both', alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig('cmc_curves.png', dpi=140); plt.show()


In [ ]:
# Precision@K sweep — useful complement to CMC
Ks = [1, 2, 4, 6, 8, 10,12, 14, 16, 18, 20]
def sweep(q, db, ql, dl):
    dist = hamming(q, db); order = dist.argsort(1)
    rel  = relevant_mask(ql, dl); rs = torch.gather(rel, 1, order)
    return [precision_at_k(rs, K) for K in Ks]

p_i2t = sweep(q_i, db_t, q_lbl_d, db_lbl_d)
p_t2i = sweep(q_t, db_i, q_lbl_d, db_lbl_d)

fig, ax = plt.subplots(figsize=(7,5))
ax.plot(Ks, p_i2t, 'o-', label='I→T'); ax.plot(Ks, p_t2i, 's-', label='T→I')
ax.set_xscale('log'); ax.set_xlabel('K'); ax.set_ylabel('Precision@K')
ax.set_title(f'Precision@K — {cfg.hash_bits}-bit codes')
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig('pk_curves.png', dpi=140); plt.show()


## Retrieve images for a text query

## 11. Save artifacts

In [ ]:
out = Path('/content/artifacts'); out.mkdir(exist_ok=True)
torch.save({'model_state': model.state_dict(),
            'cfg': cfg.__dict__,
            'history': history}, out/'cmhash_model.pt')
np.savez(out/'codes.npz',
         db_img=db_i.cpu().numpy(),  db_txt=db_t.cpu().numpy(),
         q_img=q_i.cpu().numpy(),    q_txt=q_t.cpu().numpy(),
         db_lbl=train_lbl.numpy(),   q_lbl=query_lbl.numpy())
np.savez(out/'metrics.npz',
         i2t_cmc=i2t['cmc'], t2i_cmc=t2i['cmc'],
         i2t_map=i2t['mAP@R'], t2i_map=t2i['mAP@R'],
         Ks=np.array(Ks), p_i2t=np.array(p_i2t), p_t2i=np.array(p_t2i))
print('Saved to', out)
!ls -lh {out}


In [ ]:
def retrieve_images(query: str, top_k: int = 8):
    q_code = hash_text_query(query)
    # Need continuous query code too — easy fix: drop .sign() in hash_text_query
    # OR compute it here from the same cached projection
    q_cont = q_code.float()                # interim — see Change 3 for the proper fix
    idx, h, c = retrieve_rerank(q_code, q_cont, db_i, db_ic, top_k=top_k)
    return [(int(j), float(h[i].item()), float(c[i].item()),
             db_records[j]) for i, j in enumerate(idx)]

In [ ]:
@torch.no_grad()
def hash_text_query(query: str):
    """Returns (binary_code, continuous_code) for a free-text query.
    Runs a real forward pass through the trained heterogeneous GCN."""
    model.eval()
    q_txt = embed_text(query).cpu()                          # [1, 768]

    g = HeteroData()
    n_db = graph['image'].x.size(0)
    g['image'].x = graph['image'].x.cpu()
    g['text' ].x = torch.cat([graph['text'].x.cpu(), q_txt], 0)
    g['label'].x = graph['label'].x.cpu()

    pair = torch.arange(n_db)
    g['image','pair','text'].edge_index = torch.stack([pair, pair])
    g['text','pair','image'].edge_index = torch.stack([pair, pair])

    img_e = train_lbl.nonzero(as_tuple=False).t()
    g['image','has','label'].edge_index = img_e
    g['label','of'  ,'image'].edge_index = img_e.flip(0)
    g['text' ,'has','label'].edge_index = img_e
    g['label','of'  ,'text' ].edge_index = img_e.flip(0)
    g['image','knn','image'].edge_index = knn_edges(g['image'].x, k=10)
    g['text' ,'knn','text' ].edge_index = knn_edges(g['text' ].x, k=10)

    g = g.to(DEVICE)
    _, z_t_cont, _, _ = model(g.x_dict, g.edge_index_dict)
    q_cont = z_t_cont[-1:]                                   # [1, hash_bits]
    q_bin  = q_cont.sign()
    return q_bin, q_cont

In [ ]:
def retrieve_images(query: str, top_k: int = 8):
    q_bin, q_cont = hash_text_query(query)
    idx, h, c = retrieve_rerank(q_bin, q_cont, db_i, db_ic, top_k=top_k)
    return [(int(j), float(h[i].item()), float(c[i].item()),
             db_records[j]) for i, j in enumerate(idx)]

In [ ]:
# ============================================================
# 13. Image → Text retrieval demo
# ============================================================
from PIL import Image
import matplotlib.pyplot as plt
import textwrap, torch, numpy as np
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer

# Map DB indices back to the original `records` list so we can open files
db_records = [records[i] for i in t_idx.tolist()]

# --- 13.1  Load ViT and BERT once (re-used for every query) ------------------
_vit_proc = AutoImageProcessor.from_pretrained(cfg.vit_name)
_vit_mdl  = AutoModel.from_pretrained(cfg.vit_name).to(DEVICE).eval()

_bert_tok = AutoTokenizer.from_pretrained(cfg.bert_name)
_bert_mdl = AutoModel.from_pretrained(cfg.bert_name).to(DEVICE).eval()

@torch.no_grad()
def embed_image(path_or_pil) -> torch.Tensor:
    """Raw ViT [CLS] feature for an image — shape [1, 768]."""
    img = (Image.open(path_or_pil).convert('RGB')
           if isinstance(path_or_pil, str) else path_or_pil.convert('RGB'))
    px  = _vit_proc(images=img, return_tensors='pt')['pixel_values'].to(DEVICE)
    return _vit_mdl(pixel_values=px).last_hidden_state[:, 0]

@torch.no_grad()
def embed_text(text: str) -> torch.Tensor:
    """Raw BERT [CLS] feature for a text query — shape [1, 768]."""
    enc = _bert_tok([text], padding=True, truncation=True, max_length=64,
                    return_tensors='pt').to(DEVICE)
    return _bert_mdl(**enc).last_hidden_state[:, 0]

# --- 13.2  Hash an image query through the trained model ------------
@torch.no_grad()
def hash_image_query(path_or_pil) -> torch.Tensor:
    """Encode an image query through the same modules used at training time."""
    model.eval()
    q_img = embed_image(path_or_pil).cpu()                       # [1, 768]

    # Insert the query as one extra image node into the existing graph.
    g = HeteroData()
    n_db = graph['image'].x.size(0)

    # Database features + query image
    g['image'].x = torch.cat([graph['image'].x.cpu(), q_img], 0) # +1 node
    g['text' ].x = graph['text'].x.cpu()
    g['label'].x = graph['label'].x.cpu()

    # Edges:
    # Paired cross-modal edges for DB only
    pair = torch.arange(n_db)
    g['image','pair','text'].edge_index = torch.stack([pair, pair])
    g['text','pair','image'].edge_index = torch.stack([pair, pair])

    # Label-assignment edges for DB only
    img_e = train_lbl.nonzero(as_tuple=False).t()
    g['image','has','label'].edge_index = img_e
    g['label','of'  ,'image'].edge_index = img_e.flip(0)
    g['text' ,'has','label'].edge_index = img_e
    g['label','of'  ,'text' ].edge_index = img_e.flip(0)

    # kNN intra-modal edges:
    # Recompute kNN for the *new* combined set of nodes (DB + query)
    g['image','knn','image'].edge_index = knn_edges(g['image'].x, k=10)
    g['text' ,'knn','text' ].edge_index = knn_edges(g['text' ].x, k=10)


    g = g.to(DEVICE)
    z_i, _, _, _ = model(g.x_dict, g.edge_index_dict)
    return z_i[-1:].sign()                                # [1, hash_bits]

# --- 13.3  Retrieve top-K captions ----------------------------------
def retrieve_texts(path_or_pil, top_k: int = 4):
    q_code = hash_image_query(path_or_pil)                  # [1, b]
    dist   = hamming(q_code, db_t).squeeze(0)               # [N_db]
    order  = dist.argsort()[:top_k]
    return [(int(j), float(dist[j].item()), db_records[j]) for j in order]

# --- 13.4  Visualisation -------------------------------------------
def show_image_query(path_or_pil, top_k: int = 8):
    """Show the query image on the left, ranked captions on the right."""
    hits = retrieve_texts(path_or_pil, top_k)

    fig = plt.figure(figsize=(13, max(4, 0.55 * top_k + 1)))
    gs  = fig.add_gridspec(1, 2, width_ratios=[1, 2.2], wspace=0.05)

    # Query image
    ax_img = fig.add_subplot(gs[0, 0])
    img = (Image.open(path_or_pil).convert('RGB')
           if isinstance(path_or_pil, str) else path_or_pil.convert('RGB'))
    ax_img.imshow(img); ax_img.axis('off')
    ax_img.set_title('Query image', fontsize=12)

    # Ranked captions
    ax_txt = fig.add_subplot(gs[0, 1]); ax_txt.axis('off')
    ax_txt.set_title(f'Top-{top_k} retrieved captions', fontsize=12, loc='left')
    lines = []
    for rank, (j, d, rec) in enumerate(hits, 1):
        cap   = textwrap.fill(rec['caption'], width=70,
                              subsequent_indent='        ')
        cats  = ', '.join(idx_to_cat[i]
                          for i in np.where(rec['label'] > 0)[0][:4])
        lines.append(f'#{rank:>2}  d={d:>4.0f}   {cap}\n        [{cats}]')
    ax_txt.text(0.0, 0.98, '\n\n'.join(lines), va='top', ha='left',
                family='monospace', fontsize=10, transform=ax_txt.transAxes)

    plt.tight_layout(); plt.show()
    return hits

# --- 13.5  Try it: query with a held-out database image -------------
# Pick an image from the query split (held out from training graph)
demo_idx = 8 # Changed from 50 to 51
demo_path = records[q_idx[demo_idx].item()]['file']
print('Querying with image:', demo_path)
print('True caption:', records[q_idx[demo_idx].item()]['caption'])
show_image_query(demo_path, top_k=4)

# --- 13.6  Or query with your own uploaded image -------------------
# Uncomment to upload from your local machine in Colab:
# from google.colab import files
# up = files.upload()
# for fname in up:
#     show_image_query(fname, top_k=8)


In [ ]:
# ============================================================
# 14. Text → Image retrieval demo
# ============================================================
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Map DB indices back to the original `records` so we can open image files
if 'db_records' not in globals():
    db_records = [records[i] for i in t_idx.tolist()]

# --- 14.1  Visualise the top-K retrieved images for a text query ----
def show_text_query(query: str, top_k: int = 4, cols: int = 4,
                    coarse_pool: int = 200, alpha: float = 0.7):
    """Encode a free-text query, retrieve top-K images, and display them.

    Each thumbnail shows the rank, Hamming distance (coarse), cosine on
    continuous codes (fine tie-break), and the image's mined concept tags.
    """
    # Use the existing two-stage retrieval (Hamming → cosine re-rank)
    q_bin, q_cont = hash_text_query(query)
    idx, h_dist, cos_sim = retrieve_rerank(
        q_bin, q_cont, db_i, db_ic,
        top_k=top_k, coarse_pool=coarse_pool, alpha=alpha)

    # Layout
    rows = (top_k + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.4, rows * 3.6))
    fig.suptitle(f'Text query: "{query}"', fontsize=14, y=1.01)
    axes = np.array(axes).reshape(-1)

    for r, ax in enumerate(axes):
        if r >= top_k:
            ax.axis('off'); continue
        j   = int(idx[r].item())
        rec = db_records[j]
        img = Image.open(rec['file']).convert('RGB')
        ax.imshow(img); ax.axis('off')

        cats = [idx_to_cat[i]
                for i in np.where(rec['label'] > 0)[0][:4]]
        ax.set_title(
            f'#{r+1}   H={int(h_dist[r].item())}   '
            f'cos={cos_sim[r].item():+.2f}\n'
            f'{", ".join(cats)}',
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

    # Also return the structured hits in case you want to inspect them
    return [
        {
            'rank':     r + 1,
            'db_index': int(idx[r].item()),
            'hamming':  int(h_dist[r].item()),
            'cosine':   float(cos_sim[r].item()),
            'caption':  db_records[int(idx[r].item())]['caption'],
            'file':     db_records[int(idx[r].item())]['file'],
            'concepts': [idx_to_cat[i]
                         for i in np.where(db_records[int(idx[r].item())]['label'] > 0)[0]],
        }
        for r in range(top_k)
    ]

# --- 14.2  Try several free-text queries -----------------------------
demo_queries = [
        'children are playing outdoor'
]

for q in demo_queries:
    print('=' * 80)
    print(f'QUERY: {q}')
    hits = show_text_query(q, top_k=8, cols=4, alpha=0.7)
    # Brief textual summary of the top-3 captions returned
    print('Top-3 retrieved captions:')
    for h in hits[:3]:
        print(f"  #{h['rank']}  H={h['hamming']:>2}  cos={h['cosine']:+.2f}  '{h['caption']}'")


---
### Where to extend
- **Scale up**: replace val2017 with train2017 (118 k images) and bump `n_train` / epochs.
- **Asymmetric hashing**: replace `sign()` discretisation with DCH-style proxy or HashNet's continuation.
- **Stronger graph**: add caption-token sub-graph or knowledge-graph entities as a 4th node type.
- **Triplet sampling** instead of full pairwise — necessary once the database exceeds GPU memory.
- **End-to-end**: unfreeze ViT/BERT and fine-tune jointly (drop `cache` step, stream batches).


In [ ]:
import torch
print('=== Training quality ===')
print(f"final total loss : {history['loss'][-1]:.4f}")
print(f"final nce  loss : {history['nce'][-1]:.4f}   (want < 0.3)")
print(f"final quant loss : {history['quant'][-1]:.4f}   (want < 0.2)")

print('\n=== Code health ===')
print(f"db_i  bit balance: {db_i.float().mean(0).abs().mean().item():.3f}   (want < 0.2)")
print(f"db_t  bit balance: {db_t.float().mean(0).abs().mean().item():.3f}")
print(f"unique db_i codes: {len(torch.unique(db_i, dim=0))} / {db_i.size(0)}")

print('\n=== Sanity: text\u2192image retrieval using a held-out caption ===')
# Use one of the training captions whose label is known, encoded the SAME way
# the model saw it during training (no kNN re-attachment).
probe_idx = 0
probe_code  = db_t[probe_idx:probe_idx+1]                       # paired text code
# Ensure train_lbl is on the same device as 'order'
probe_label = train_lbl.to(DEVICE)[probe_idx]
dist  = hamming(probe_code, db_i).squeeze(0)
order = dist.argsort()[:10]
# Ensure train_lbl is on the same device as 'order'
shared = (train_lbl.to(DEVICE)[order] * probe_label).sum(1) > 0
print(f"probe caption: {db_records[probe_idx]['caption']!r}")
print(f"top-10 share \u22651 label with query: {shared.sum().item()}/10")
print(f"top-10 distances: {dist[order].tolist()}")

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def loss_breakdown_at_convergence(model, graph, train_lbl_d):
    model.eval()
    z_i, z_t, lo_i, lo_t = model(graph.x_dict, graph.edge_index_dict)
    y = train_lbl_d

    L_nce  = info_nce(z_i, z_t, temp=0.2)
    L_sup  = supervised_contrastive(z_i, y, temp=0.2) + \
             supervised_contrastive(z_t, y, temp=0.2)
    # L_xsup = supervised_contrastive_xm(z_i, z_t, y, temp=0.2) # This function is not defined
    L_q    = quant_loss(z_i) + quant_loss(z_t)
    L_b    = balance_loss(z_i) + balance_loss(z_t)
    L_d    = decorrelation_loss(z_i) + decorrelation_loss(z_t)
    L_c    = (F.binary_cross_entropy_with_logits(lo_i, y) +
              F.binary_cross_entropy_with_logits(lo_t, y))

    weights = {'nce':1.0,'sup':3.0,'quant':0.05,
               'bal':1.0,'dec':1.0,'cls':1.0}
    raw = {'nce':L_nce.item(),'sup':L_sup.item(),
           'quant':L_q.item(),'bal':L_b.item(),'dec':L_d.item(),'cls':L_c.item()}

    print(f'{str("loss"):<8}{str("raw"):>10}{str("x weight"):>12}{str("% of total"):>14}')
    print('-' * 44)
    weighted = {k: raw[k] * weights[k] for k in raw}
    total = sum(weighted.values())
    for k in sorted(raw, key=lambda x: -weighted[x]):
        print(f'{k:<8}{raw[k]:>10.3f}{weighted[k]:>12.2f}'
              f'{weighted[k]/total*100:>13.1f}%')
    print('-' * 44)
    print(f'{str("total"):<8}{str(""):>10}{total:>12.2f}')

loss_breakdown_at_convergence(model, graph, train_lbl_d)